<a href="https://colab.research.google.com/github/jaysulk/Poisson-GENERIC-Neural-Operators/blob/main/ICLR2027_learned_symbols.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learned symbols of the Transolver PG-phys models (no training)

Part A of `ICLR2027_camera_ready.ipynb` saved every trained model to Drive (`pg_camera_ready/ckpt/`). This notebook only reads the learned operator symbols out of those checkpoints: the transport coefficient α (advection, exact −cΔt = −0.1), the wave symbol d(k) on the data band (exact Δt = 0.1) and the Lie–Poisson coefficient λ (Burgers, exact −Δt/3 = −0.0333). It runs in under a minute on CPU.

Output: one table per equation, then the figures for the sentence in Section 4 ("the learned symbols recover the exact ones …") and for Remark 2, for all backbones found and for the Transolver runs alone. Also written to `pg_camera_ready/learned_symbols.csv`. The FNO models of the integrator study are included if they are on Drive; CNO models were never saved, so they do not appear.

In [1]:
# ===== Learned symbols of the trained PG-phys models, read from the checkpoints already on Drive (no training) =====
import os, re, glob, math
import numpy as np, pandas as pd, torch

FOLDERS = ['/content/drive/MyDrive/pg_camera_ready/ckpt', '/content/drive/MyDrive/pg_integrators/ckpt']
DT, C_ADV = 0.1, 1.0                         # frame spacing and advection speed of the data
KMAX = {1: 8, 2: 6}                          # data band |k|_inf <= KMAX
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print(f'Drive not mounted ({type(e).__name__}); looking in FOLDERS as given')

def wave_symbol(a, dim):
    """d(k) = (a(k)^2 + a(-k)^2) / 2 on the data band, k != 0 (PoissonOperator.symbol, canonical branch)."""
    m = (a.shape[-1] - 1) // 2; d2 = a.double() ** 2
    d = 0.5 * (d2 + torch.flip(d2, tuple(range(dim))))                 # index i <-> k = i - m, so the flip is k -> -k
    k = torch.arange(-m, m + 1)
    kinf = k.abs() if dim == 1 else torch.maximum(k.abs()[:, None], k.abs()[None, :])
    return d[(kinf > 0) & (kinf <= KMAX[dim])]

rows = []
for folder in FOLDERS:
    for path in sorted(glob.glob(f'{folder}/*.pt')):
        name = os.path.basename(path)[:-3]
        mt = re.fullmatch(r'(\d)D_(heat|advection|burgers|wave)_(?:(fno|transolver|cno)_)?(pgphys)_(\d+)', name)
        if not mt: continue                                             # baselines, ablations and tagged variants are skipped
        dim, pde, bb, _, seed = int(mt[1]), mt[2], mt[3] or 'fno', mt[4], int(mt[5])
        sd = torch.load(path, map_location='cpu')
        alpha, lam, a = sd['L.alpha'][0].item(), sd['L.lam'].item(), sd['L.a']
        ok = all(torch.isfinite(v).all().item() for v in sd.values() if torch.is_floating_point(v))
        r = dict(dim=dim, pde=pde, backbone=bb, seed=seed, finite=ok, file=name)
        if ok and pde == 'advection':
            ex = -C_ADV * DT; r.update(alpha=alpha, alpha_exact=ex, alpha_err_pct=100 * abs(alpha - ex) / abs(ex),
                                       symbol_abs_err_max=abs(alpha - ex) * KMAX[dim] * dim, lam=lam)
        if ok and pde == 'burgers':
            ex = -DT / 3; r.update(lam=lam, lam_exact=ex, lam_err_pct=100 * abs(lam - ex) / abs(ex), alpha=alpha)
        if ok and pde == 'heat': r.update(alpha=alpha, lam=lam)        # exact: both zero
        if ok and pde == 'wave':
            d = wave_symbol(a, dim); r.update(d_min=d.min().item(), d_max=d.max().item(), d_exact=DT, d_err_pct=100 * (d - DT).abs().max().item() / DT)
        rows.append(r)
df = pd.DataFrame(rows).drop_duplicates(['dim', 'pde', 'backbone', 'seed']).sort_values(['dim', 'pde', 'backbone', 'seed'])
assert len(df), 'no PG-phys checkpoints found in ' + ', '.join(FOLDERS)
out = os.path.dirname(FOLDERS[0]) + '/learned_symbols.csv'; df.to_csv(out, index=False)
with pd.option_context('display.width', 220, 'display.max_columns', 30, 'display.float_format', lambda x: f'{x:.5g}'):
    for pde, cols in [('advection', ['alpha', 'alpha_exact', 'alpha_err_pct', 'symbol_abs_err_max', 'lam']), ('burgers', ['lam', 'lam_exact', 'lam_err_pct', 'alpha']),
                      ('wave', ['d_min', 'd_max', 'd_exact', 'd_err_pct']), ('heat', ['alpha', 'lam'])]:
        g = df[df.pde == pde]
        if len(g): print(f'\n--- {pde} ---'); print(g[['dim', 'backbone', 'seed', 'finite'] + [c for c in cols if c in g]].to_string(index=False))

print('\n================ for the sentence in Section 4 and Remark 2 ================')
def grp(pde, dim=None):
    g = df[(df.pde == pde) & df.finite]; return g if dim is None else g[g.dim == dim]
for label, sel in [('all backbones found', lambda g: g), ('Transolver only', lambda g: g[g.backbone == 'transolver'])]:
    print(f'\n[{label}]')
    g = sel(grp('advection', 1))
    if len(g): print(f'  1D advection, {len(g)} runs: alpha in [{g.alpha.min():.5f}, {g.alpha.max():.5f}] (exact {-C_ADV * DT:.5f}); largest |alpha k + c k dt| on the data band {g.symbol_abs_err_max.max():.1e}; largest relative error {g.alpha_err_pct.max():.3f}%')
    g = sel(grp('advection', 2))
    if len(g): print(f'  2D advection, {len(g)} runs: alpha in [{g.alpha.min():.5f}, {g.alpha.max():.5f}]; largest relative error {g.alpha_err_pct.max():.3f}%')
    g = sel(grp('wave', 1))
    if len(g): print(f'  1D wave, {len(g)} runs: d(k) in [{g.d_min.min():.4f}, {g.d_max.max():.4f}] (exact {DT}); largest relative error {g.d_err_pct.max():.2f}%')
    g = sel(grp('wave', 2))
    if len(g): print(f'  2D wave, {len(g)} runs: d(k) in [{g.d_min.min():.4f}, {g.d_max.max():.4f}]; largest relative error {g.d_err_pct.max():.2f}%')
    g = sel(grp('burgers'))
    if len(g): print(f'  Burgers, {len(g)} runs (1D and 2D): lambda = {g.lam.mean():.4f} +- {g.lam.std(ddof=1) if len(g) > 1 else 0:.4f} (exact {-DT / 3:.4f}); range [{g.lam.min():.4f}, {g.lam.max():.4f}]; largest relative error {g.lam_err_pct.max():.2f}%')
bad = df[~df.finite]
if len(bad): print('\nnot finite (diverged in training), left out:', ', '.join(bad.file))
print('\nwrote', out)

Mounted at /content/drive

--- advection ---
 dim   backbone  seed  finite     alpha  alpha_exact  alpha_err_pct  symbol_abs_err_max         lam
   1        fno     0    True  -0.10002         -0.1       0.016259          0.00013007 -0.00034781
   1        fno     1    True  -0.10003         -0.1       0.025669          0.00020535  0.00032874
   1        fno     2    True  -0.10002         -0.1       0.016929          0.00013543  0.00022696
   1 transolver     0    True  -0.10001         -0.1      0.0099331          7.9465e-05   4.311e-05
   1 transolver     1    True  -0.10006         -0.1       0.057147          0.00045718  0.00036122
   1 transolver     2    True  -0.10003         -0.1       0.030042          0.00024034  0.00014746
   2        fno     0    True -0.099995         -0.1      0.0054225           6.507e-05  0.00016386
   2 transolver     0    True -0.099981         -0.1       0.019296          0.00023155  0.00025277
   2 transolver     1    True  -0.10002         -0.1   